# 05 — State & Memory: mem0 Memory Agent

**Module 4 of the workshop.** Real persistent memory via mem0 + local Ollama embeddings + FAISS — the production-grade version of the store/retrieve pattern from the previous notebook. This one took real debugging to get working locally; read the config notes before running.


## Problem

An LLM call is stateless by default — ask it something, get an answer, ask again with no memory of the first. Real assistants need to remember facts across separate invocations, not just within one conversation, and need that memory to be *searchable by meaning*, not just recency.


## Concept

```
Agent
 ├── Conversation state   (within one session)
 ├── Session               (one user's ongoing interaction)
 └── Long-term memory       (persists across sessions entirely)
```

Store: content gets embedded (turned into a vector) and saved to FAISS. Retrieve: the query gets embedded too, and the store returns whatever's closest by similarity — this is why the *embedding model* matters as much as the chat model.

**When would you NOT use this?** If you just need the last N messages of *this* conversation, that's session state, not memory — far simpler, no embeddings, no vector store. mem0 earns its cost when facts need to survive across sessions or be searched by meaning.


## Why this bypasses `strands_tools.mem0_memory`

`strands_tools.mem0_memory` (v0.8.9) has a real bug: its FAISS backend hardcodes `embedding_model_dims=1536/1024` depending on code path, ignoring `MEM0_EMBEDDER_MODEL`, while the embedder itself and mem0's own internal defaults use yet other dims (512/768) — three inconsistent values inside one call chain. No single env var or monkeypatch reconciles it reliably.

This script constructs `mem0.Memory` directly instead, with an explicit config where every dim matches `nomic-embed-text`'s real 768-dim output. mem0's own internal LLM/embedder stay pinned to local Ollama regardless of which chat model backs the outer agent — mem0 has no Bedrock provider wired up here, and this keeps memory storage working offline even when the chat model itself is cloud-backed.


## Architecture

```
"Remember that my favorite language is Rust."
              │
              ▼
     ┌──────────────────────┐
     │ MemoryAssistant       │
     │ .process_input()      │
     └──────────┬────────────┘
                │ starts with "remember "/"note that "/"i want you to know "
                ▼
     ┌──────────────────────┐
     │ mem0.Memory.add()      │──▶ embeds via nomic-embed-text (Ollama)
     │ (self.memory)          │──▶ stores vector in FAISS (mem0_data/faiss)
     └────────────────────────┘
                │
                ▼
     "I've stored that information in my memory."


"What's my favorite language?"
              │
              ▼
     ┌──────────────────────┐
     │ MemoryAssistant       │
     │ .process_input()      │
     └──────────┬────────────┘
                │ doesn't match a "remember" prefix → retrieve path
                ▼
     ┌──────────────────────┐
     │ mem0.Memory.search()   │──▶ embeds query, finds closest vector
     └──────────┬────────────┘
                │ context = matched memories (score >= min_score)
                ▼
     ┌──────────────────────┐
     │ answerer Agent         │  (callback_handler=None,
     │ (self.agent unused     │   system_prompt: "use known info if relevant")
     │  for retrieval path)   │
     └──────────┬────────────┘
                │
                ▼
        "Rust." ──────────▶ user
```


## Step 1 — Resolve the model and set up the mem0 config

Note the dimension consistency: `embedding_model_dims=768` in the vector store config matches `embedding_dims=768` in the embedder config — both tied to `nomic-embed-text`'s real output size. This is the exact mismatch that breaks the packaged `strands_tools.mem0_memory` tool.


In [1]:
import sys
from pathlib import Path
from typing import Any

sys.path.insert(0, str(Path.cwd().parent))

from mem0 import Memory
from model_provider import OLLAMA_HOST, get_model
from strands import Agent

model = get_model()
print(f"Using: {type(model).__name__}")

MEM0_CONFIG = {
    "vector_store": {
        "provider": "faiss",
        "config": {"embedding_model_dims": 768, "path": "mem0_data/faiss"},
    },
    "embedder": {
        "provider": "ollama",
        "config": {
            "model": "nomic-embed-text",
            "ollama_base_url": OLLAMA_HOST,
            "embedding_dims": 768,
        },
    },
    "llm": {
        "provider": "ollama",
        "config": {
            "model": "qwen3.5:4b",
            "ollama_base_url": OLLAMA_HOST,
            "temperature": 0.1,
            "max_tokens": 2000,
        },
    },
}

MEMORY_SYSTEM_PROMPT = "You manage a user's personal memory store precisely and concisely."


Using: OllamaModel


## Step 2 — Build the `MemoryAssistant` class

Wraps `mem0.Memory` (the actual vector store) and a Strands `Agent` (used to phrase the final retrieval answer) behind one small interface: `store_memory`, `retrieve_memories`, `process_input`. `process_input` routes store vs retrieve by checking for a "remember"/"note that" prefix — simpler than the classifier-agent approach from the previous notebook.


In [2]:
class MemoryAssistant:
    def __init__(self, user_id: str = "demo_user"):
        self.user_id = user_id
        self.memory = Memory.from_config(MEM0_CONFIG)
        self.agent = Agent(model=model, system_prompt=MEMORY_SYSTEM_PROMPT)

    def store_memory(self, content: str) -> dict[str, Any]:
        return self.memory.add(content, user_id=self.user_id)

    def retrieve_memories(self, query: str, min_score: float = 0.3, max_results: int = 5):
        results = self.memory.search(query, user_id=self.user_id, limit=max_results)
        return [r for r in results.get("results", []) if r.get("score", 0) >= min_score]

    def process_input(self, user_input: str) -> str:
        lowered = user_input.lower()
        if lowered.startswith(("remember ", "note that ", "i want you to know ")):
            content = user_input.split(" ", 1)[1]
            self.store_memory(content)
            return "I've stored that information in my memory."

        memories = self.retrieve_memories(user_input)
        context = "\n".join(m["memory"] for m in memories) or "(nothing relevant stored)"
        answerer = Agent(
            model=model,
            system_prompt="Answer using the known info if relevant.",
            callback_handler=None,
        )
        return str(answerer(f"Known info: {context}\n\nUser: {user_input}"))


## Step 4 — Run it: store a fact, then retrieve it by meaning

The second call is a paraphrase ("What's my favorite language?"), not a repeat of the exact stored sentence — this is what "searchable by meaning" actually looks like.


In [3]:
assistant = MemoryAssistant(user_id="alex")
print(assistant.process_input("Remember that my favorite language is Rust."))
print(assistant.process_input("What's my favorite language?"))

[PostHog] Multiple active PostHog clients detected for the same project API key and host. Reuse one Posthog instance per app or process when possible to avoid competing background queues and missed shutdown flushes. Multiple clients are supported when intentional.


I've stored that information in my memory.
Based on the information provided, your favorite programming language is Rust.



## Failure mode to know about

This demo is intentionally not hands-on at scale — mem0 setup fragility (the dimension-mismatch issue above) makes it risky to live-debug in front of a room. If retrieval returns "nothing relevant" even though you just stored the fact, check `min_score` first (too strict a threshold silently discards a real match) before assuming the store itself is broken.
